# Prediksi dengan YOLOv8s dan RetinaNet


In [ ]:
import torch
import torchvision
import torchvision.transforms.functional as TF
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw, ImageFont
from ultralytics import YOLO

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
YOLO_WEIGHT_PATH   = 'runs/yolov8s_kardio/weights/best.pt'
RETINA_WEIGHT_PATH = 'outputs/retinanet_kardio_best.pt'
IMAGE_PATHS = [
    'dataset/test/images/04-HARLY-SYAHFRIL-10-12-1984_PNG.rf.fb2f3520aedc767924e53ac833a26b6d.jpg',
    'dataset/test/images/MCU002-normal_png.rf.9a42e3bb8ce3ebc8af69b2456e1fe94e.jpg',
]

RETINA_CLASS_NAMES = ['kardiomegali', 'normal']
YOLO_CONF          = 0.25
YOLO_IOU           = 0.45
RETINA_CONF        = 0.50

In [ ]:
# Load model YOLOv8s
yolo_model = YOLO(YOLO_WEIGHT_PATH)
print(f' YOLOv8s loaded: {YOLO_WEIGHT_PATH}')

In [ ]:
# Load model RetinaNet
def load_retinanet(path):
    checkpoint  = torch.load(path, map_location=DEVICE)

    # Handle berbagai format checkpoint
    if isinstance(checkpoint, dict) and 'model' in checkpoint and not any(
        k.startswith(('backbone', 'head')) for k in checkpoint.keys()
    ):
        state_dict = checkpoint['model']
    elif isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
    else:
        state_dict = checkpoint

    # Hitung jumlah class dari checkpoint
    key = 'head.classification_head.cls_logits.weight'
    num_classes = state_dict[key].shape[0] // 9 if key in state_dict else 91
    print(f'   Jumlah class terdeteksi: {num_classes} (termasuk background)')

    # Coba dua varian arsitektur
    for variant in ['retinanet_resnet50_fpn', 'retinanet_resnet50_fpn_v2']:
        try:
            if variant == 'retinanet_resnet50_fpn_v2':
                model = torchvision.models.detection.retinanet_resnet50_fpn_v2(
                    weights=None, weights_backbone=None, num_classes=num_classes
                )
            else:
                model = torchvision.models.detection.retinanet_resnet50_fpn(
                    weights=None, weights_backbone=None, num_classes=num_classes
                )
            model.load_state_dict(state_dict)
            model.to(DEVICE)
            model.eval()
            print(f'   Arsitektur cocok: {variant}')
            return model
        except RuntimeError:
            continue

    raise RuntimeError('Gagal load RetinaNet — cek path dan arsitektur checkpoint.')

retina_model = load_retinanet(RETINA_WEIGHT_PATH)
print(f' RetinaNet loaded: {RETINA_WEIGHT_PATH}')

In [ ]:
# Fungsi inferensi
def predict_yolo(model, image_path, conf=0.25, iou=0.45):
    results = model.predict(source=image_path, conf=conf, iou=iou, verbose=False)
    r = results[0]
    annotated = Image.fromarray(r.plot()[:, :, ::-1])
    detections = [
        {
            'class': model.names[int(b.cls[0])],
            'score': round(float(b.conf[0]), 3),
            'box'  : [round(v, 1) for v in b.xyxy[0].tolist()]
        }
        for b in r.boxes
    ]
    return annotated, detections


def get_color(label):
    rng = np.random.default_rng(label * 9973 + 17)
    return tuple(int(x) for x in rng.integers(50, 230, size=3))


def predict_retinanet(model, image_path, conf=0.5, class_names=None):
    image      = Image.open(image_path).convert('RGB')
    img_tensor = TF.to_tensor(image).to(DEVICE)

    with torch.no_grad():
        preds = model([img_tensor])[0]

    boxes  = preds['boxes'].cpu().numpy()
    labels = preds['labels'].cpu().numpy()
    scores = preds['scores'].cpu().numpy()

    annotated  = image.copy()
    draw       = ImageDraw.Draw(annotated)
    font_size  = max(16, int(min(image.size) * 0.03))
    try:
        font = ImageFont.truetype('arial.ttf', font_size)
    except Exception:
        font = ImageFont.load_default()

    detections = []
    for box, label, score in zip(boxes, labels, scores):
        if score < conf:
            continue
        label    = int(label)
        cls_name = class_names[label-1] if class_names and 1 <= label <= len(class_names) else f'class_{label}'
        color    = get_color(label)
        x1, y1, x2, y2 = [float(v) for v in box]
        bw = max(2, int(min(image.size) * 0.006))
        draw.rectangle([x1, y1, x2, y2], outline=color, width=bw)
        caption = f'{cls_name} {score:.2f}'
        tb = draw.textbbox((x1, y1), caption, font=font)
        draw.rectangle((tb[0]-2, tb[1]-2, tb[2]+2, tb[3]+2), fill=color)
        draw.text((x1, y1), caption, fill='black', font=font)
        detections.append({'class': cls_name, 'score': round(float(score), 3), 'box': [round(v,1) for v in box]})

    return annotated, detections

In [ ]:
# Prediksi & tampilkan hasil 
for IMAGE_PATH in IMAGE_PATHS:
    yolo_img,   yolo_dets   = predict_yolo(yolo_model, IMAGE_PATH, YOLO_CONF, YOLO_IOU)
    retina_img, retina_dets = predict_retinanet(retina_model, IMAGE_PATH, RETINA_CONF, RETINA_CLASS_NAMES)
    original = Image.open(IMAGE_PATH).convert('RGB')

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(original);   axes[0].set_title('Gambar Asli', fontsize=14);                    axes[0].axis('off')
    axes[1].imshow(yolo_img);   axes[1].set_title(f'YOLOv8s ({len(yolo_dets)} deteksi)', fontsize=14);   axes[1].axis('off')
    axes[2].imshow(retina_img); axes[2].set_title(f'RetinaNet ({len(retina_dets)} deteksi)', fontsize=14); axes[2].axis('off')

    plt.suptitle(f'Hasil Prediksi — {IMAGE_PATH}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f'\n Deteksi YOLOv8s:')
    for d in yolo_dets:   print(f'  class={d["class"]}  score={d["score"]}  box={d["box"]}')

    print(f'\n Deteksi RetinaNet:')
    for d in retina_dets: print(f'  class={d["class"]}  score={d["score"]}  box={d["box"]}')

In [ ]:
# Test banyak gambar sekaligus
import os
import random

TEST_DIR = 'dataset/test/images'
EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp')

image_paths = [
    os.path.join(TEST_DIR, f)
    for f in os.listdir(TEST_DIR)
    if f.lower().endswith(EXTENSIONS)
]
print(f'Total gambar ditemukan: {len(image_paths)}')

random.shuffle(image_paths) 

for img_path in image_paths[:3]:
    yolo_img,   yolo_dets   = predict_yolo(yolo_model, img_path, YOLO_CONF, YOLO_IOU)
    retina_img, retina_dets = predict_retinanet(retina_model, img_path, RETINA_CONF, RETINA_CLASS_NAMES)
    original = Image.open(img_path).convert('RGB')

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].imshow(original);   axes[0].set_title('Asli');                            axes[0].axis('off')
    axes[1].imshow(yolo_img);   axes[1].set_title(f'YOLO ({len(yolo_dets)})');        axes[1].axis('off')
    axes[2].imshow(retina_img); axes[2].set_title(f'RetinaNet ({len(retina_dets)})'); axes[2].axis('off')
    plt.suptitle(os.path.basename(img_path), fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# Test banyak gambar teratas sekaligus
import os

TEST_DIR = 'dataset/test/images'
EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp')

image_paths = [
    os.path.join(TEST_DIR, f)
    for f in os.listdir(TEST_DIR)
    if f.lower().endswith(EXTENSIONS)
]
print(f'Total gambar ditemukan: {len(image_paths)}')

for img_path in image_paths[:3]:
    yolo_img,   yolo_dets   = predict_yolo(yolo_model, img_path, YOLO_CONF, YOLO_IOU)
    retina_img, retina_dets = predict_retinanet(retina_model, img_path, RETINA_CONF, RETINA_CLASS_NAMES)
    original = Image.open(img_path).convert('RGB')

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].imshow(original);   axes[0].set_title('Asli');     axes[0].axis('off')
    axes[1].imshow(yolo_img);   axes[1].set_title(f'YOLO ({len(yolo_dets)})');   axes[1].axis('off')
    axes[2].imshow(retina_img); axes[2].set_title(f'RetinaNet ({len(retina_dets)})'); axes[2].axis('off')
    plt.suptitle(os.path.basename(img_path), fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()